# 02 — Exploratory Data Analysis
Distributions, class imbalance, correlations, and outlier detection.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from src.config import DATA_PROCESSED_PATH, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
os.makedirs('docs/eda_plots', exist_ok=True)

df = pd.read_csv(DATA_PROCESSED_PATH)
print(f"Loaded {df.shape[0]:,} rows")

## 1. Numerical Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_FEATURES):
    df[col].hist(bins=50, ax=axes[i], edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.savefig('docs/eda_plots/hist_numerical.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**
- `Income` and `LoanAmount` span wide ranges with roughly uniform distribution
- `CreditScore` ranges 300–850 with even spread
- `DTIRatio` is uniformly distributed between 0.0 and 1.0
- `NumCreditLines` has only values 1–4 (discrete)

## 2. Class Imbalance Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
counts = df[TARGET].value_counts()
total = len(df)
bars = ax.bar(['No Default (0)', 'Default (1)'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='black')
for bar, count in zip(bars, counts.values):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 500,
            f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=14)
ax.set_title('Class Distribution — Default', fontsize=16, fontweight='bold')
ax.set_ylabel('Count')
plt.savefig('docs/eda_plots/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Boxplots — Income and LoanAmount by Default Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, col in enumerate(['Income', 'LoanAmount']):
    sns.boxplot(data=df, x=TARGET, y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} by Default Status', fontsize=14, fontweight='bold')
    axes[i].set_xticklabels(['No Default', 'Default'])
plt.tight_layout()
plt.savefig('docs/eda_plots/boxplot_income.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df[NUMERICAL_FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.savefig('docs/eda_plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, col in enumerate(['Education', 'LoanPurpose', 'EmploymentType']):
    df[col].value_counts().plot(kind='barh', ax=axes[i], color='#3498db', edgecolor='black')
    axes[i].set_title(f'{col} — Value Counts', fontsize=14, fontweight='bold')
    axes[i].invert_yaxis()
plt.tight_layout()
plt.savefig('docs/eda_plots/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key EDA Findings
1. **Class Imbalance:** ~88% No Default vs ~12% Default — 7.6:1 ratio
2. **No Missing Values:** Dataset is clean, but we keep imputers in pipeline for safety
3. **Feature Ranges:** Income (15K–150K), LoanAmount (5K–250K), CreditScore (300–850)
4. **Low Correlation to Target:** No single feature strongly predicts default
5. **Categorical Balance:** Education and EmploymentType categories are roughly balanced